# MCP Tutorial

In this tutorial, we will explore the concepts and applications of the MCP (model context protocol). 

We will write an MCP server, a local mcp clint to get a taste of what a client looks like. Then, we'll look at MCP Inspector to understand how to inspect and interact with the server.

In [2]:
import asyncio
import importlib.metadata
import json
import shutil
import sys
from pathlib import Path

DEMO = Path("mcp-demo")            # everything the notebook writes lives here
WORKSPACE = DEMO / "workspace"     # the only directory edit may touch
OUTSIDE = DEMO / "outside"         # step 8 writes here, outside the workspace
(WORKSPACE / "src").mkdir(parents=True, exist_ok=True)

print(sys.executable)
print("mcp", importlib.metadata.version("mcp"))

/Users/rkrsn/COMSE6998-019/lectures/lecture-5--2026-10-07/.venv/bin/python
mcp 2.3.0


In [3]:
target = WORKSPACE / "src" / "parse.py"
target.write_text(
    "def total(xs, n):\n"
    "    s, i = 0, 0\n"
    "    while i <= n:\n"
    "        s += xs[i]\n"
    "        i += 1\n"
    "    return s\n"
)
(DEMO / "slow.done").unlink(missing_ok=True)
shutil.rmtree(OUTSIDE, ignore_errors=True)
print(target.read_text())

def total(xs, n):
    s, i = 0, 0
    while i <= n:
        s += xs[i]
        i += 1
    return s



## 1. The server

- **An MCP server** is a program that offers tools (slide 30).
- **It runs as its own process,** and the host talks to it over stdin and stdout (slide 69).
- **The cells below build `mcp-demo/server.py`** one piece at a time: the first cell creates the file, and each later cell appends to it.
- **Run each cell once, in order.** Running the first cell again starts the file over.

### Imports

- **`MCPServer`** is the SDK's server class.
- **`ToolError`** is the exception a tool raises to refuse a request with a reason the model can read.
- **`ToolAnnotations`** carries the four hints on slide 63.

In [11]:
%%writefile mcp-demo/server.py
import time
from pathlib import Path
from typing import TypedDict

from mcp.server import MCPServer                                # <-- The SDK's main server class
from mcp.server.mcpserver.exceptions import ToolError           # <-- Exception raised when a tool errors out
from mcp.types import ToolAnnotations                           # <-- Used to provide metadata about the tool

Overwriting mcp-demo/server.py


### The server and its workspace

- **The name `workspace`** reaches the client in the server's info when it connects.
- **`WORKSPACE`** is the only directory `edit` may touch.
- **Both paths come from the file's own location,** so the server finds them whether the notebook or the Inspector starts it.

In [12]:
%%writefile -a mcp-demo/server.py

HERE = Path(__file__).resolve().parent
WORKSPACE = HERE / "workspace"

mcp = MCPServer("workspace")

Appending to mcp-demo/server.py


### The result type

- **`EditResult`** describes what `edit` returns.
- **The SDK turns it into** the tool's `outputSchema` (slide 54).
- **The client receives** the returned dictionary as `structuredContent`.

In [13]:
%%writefile -a mcp-demo/server.py


class EditResult(TypedDict):
    path: str
    replaced: int

Appending to mcp-demo/server.py


### The contract of `edit`

- **The decorator** holds everything the host and the model learn about the tool.
- **`name`, `title` and `description`** say what the tool is and what it does, and the model reads the description as plain text.
- **`annotations`** carry four hints about side effects, which the schema says are not guaranteed to be faithful (slide 63).
- **`meta`** is extension metadata for the host.
- **`structured_output=True`** makes the SDK publish `outputSchema` from the return type.
- **`tools/list`** returns these fields as slide 55 shows.
- **Nothing in this cell** constrains what the function below does.

In [14]:
%%writefile -a mcp-demo/server.py


@mcp.tool(
    name="edit",
    title="Edit a workspace file",
    description="Replace one exact occurrence of old with new.",
    annotations=ToolAnnotations(
        title="Edit a workspace file",
        read_only_hint=False, destructive_hint=True,
        idempotent_hint=False, open_world_hint=False),
    meta={"course": "COMS 6998-019"},
    structured_output=True,
)

Appending to mcp-demo/server.py


### The implementation of `edit`

- **The type hints** on `path`, `old` and `new` become the `inputSchema`.
- **The SDK validates every call** against that schema before the function runs (slide 79).
- **The first check** refuses a path that resolves outside the workspace.
- **The second check** refuses an `old` that does not appear exactly once.
- **Each refusal raises `ToolError`,** so the model receives the reason in a result marked `isError: true`.
- **Any other exception** reaches the model only as "Error executing tool edit".

In [ ]:
%%writefile -a mcp-demo/server.py
def edit(path: str, old: str, new: str) -> EditResult:
    target = (WORKSPACE / path).resolve()
    
    # If the target is not within the workspace, refuse the operation
    if not target.is_relative_to(WORKSPACE):
        raise ToolError(f"refused: {path} is outside the workspace")
    
    text = target.read_text()
    
    # If the old string does not appear exactly once, refuse the operation
    if text.count(old) != 1:
        raise ToolError(f"refused: old appears {text.count(old)} times in {path}")
    
    # Perform the replacement and return the result
    target.write_text(text.replace(old, new, 1))
    return {"path": path, "replaced": 1}

Appending to mcp-demo/server.py


### A slow tool

- **`slow`** blocks in `time.sleep`, which is synchronous code.
- **When the client stops waiting,** the sleep keeps running.
- **The file `slow.done`** shows that the server finished anyway.
- **With no `description=`,** the SDK takes the description from the docstring.

In [ ]:
%%writefile -a mcp-demo/server.py


@mcp.tool()
def slow(seconds: int) -> str:
    """Block for the given number of seconds, then report."""
    time.sleep(seconds)
    (HERE / "slow.done").write_text(f"slept {seconds} s\n")
    return f"slept {seconds} s"

### Running the server

- **`mcp.run()`** serves over stdio by default.
- **The guard** matters because `mcp dev` imports the file to find the server before it starts it.

In [ ]:
%%writefile -a mcp-demo/server.py


if __name__ == "__main__":
    mcp.run()

In [ ]:
source = (DEMO / "server.py").read_text()
compile(source, "server.py", "exec")
assert source.count("@mcp.tool(") == 2 and source.count("mcp.run()") == 1, "run each server cell once, in order"
print(f"server.py is complete: {len(source.splitlines())} lines")

## 2. Two launchers

- **`server_stdio.py`** serves over stdio, so a host or the Inspector starts it as a subprocess.
- **`server_http.py`** serves over Streamable HTTP at `http://127.0.0.1:8000/mcp`. Start it yourself: `python mcp-demo/server_http.py`.
- **Both files hold the whole server,** so each launches on its own and both can run at the same time.
- **`progress_bar`** reports progress after a random pause between steps, through `ctx.report_progress`.

In [ ]:
%%writefile mcp-demo/server_stdio.py
import asyncio
import random
import time
from pathlib import Path
from typing import TypedDict

from mcp.server import MCPServer
from mcp.server.mcpserver import Context                                # <-- The SDK's main server class
from mcp.server.mcpserver.exceptions import ToolError           # <-- Exception raised when a tool errors out
from mcp.types import ToolAnnotations                           # <-- Used to provide metadata about the tool

HERE = Path(__file__).resolve().parent
WORKSPACE = HERE / "workspace"

mcp = MCPServer("workspace")


class EditResult(TypedDict):
    path: str
    replaced: int


@mcp.tool(
    name="edit",
    title="Edit a workspace file",
    description="Replace one exact occurrence of old with new.",
    annotations=ToolAnnotations(
        title="Edit a workspace file",
        read_only_hint=False, destructive_hint=True,
        idempotent_hint=False, open_world_hint=False),
    meta={"course": "COMS 6998-019"},
    structured_output=True,
)
def edit(path: str, old: str, new: str) -> EditResult:
    target = (WORKSPACE / path).resolve()
    if not target.is_relative_to(WORKSPACE):
        raise ToolError(f"refused: {path} is outside the workspace")
    text = target.read_text()
    if text.count(old) != 1:
        raise ToolError(f"refused: old appears {text.count(old)} times in {path}")
    target.write_text(text.replace(old, new, 1))
    return {"path": path, "replaced": 1}


@mcp.tool(name="progress_bar", description="Report progress in steps, pausing a random time between steps.")
async def progress_bar(ctx: Context, steps: int = 10, max_gap: float = 1.5) -> str:
    for i in range(1, steps + 1):
        await asyncio.sleep(random.uniform(0.1, max_gap))             # random time gap
        bar = "#" * i + "." * (steps - i)
        await ctx.report_progress(i, steps, f"[{bar}] {i}/{steps}")   # progress notification
    return f"done in {steps} steps"


if __name__ == "__main__":
    mcp.run()                      # stdio, the default: the host starts this process

In [ ]:
%%writefile mcp-demo/server_http.py
import asyncio
import random
import time
from pathlib import Path
from typing import TypedDict

from mcp.server import MCPServer
from mcp.server.mcpserver import Context                                # <-- The SDK's main server class
from mcp.server.mcpserver.exceptions import ToolError           # <-- Exception raised when a tool errors out
from mcp.types import ToolAnnotations                           # <-- Used to provide metadata about the tool

HERE = Path(__file__).resolve().parent
WORKSPACE = HERE / "workspace"

mcp = MCPServer("workspace")


class EditResult(TypedDict):
    path: str
    replaced: int


@mcp.tool(
    name="edit (http)",
    title="Edit a workspace file",
    description="Replace one exact occurrence of old with new.",
    annotations=ToolAnnotations(
        title="Edit a workspace file",
        read_only_hint=False, destructive_hint=True,
        idempotent_hint=False, open_world_hint=False),
    meta={"course": "COMS 6998-019"},
    structured_output=True,
)
def edit(path: str, old: str, new: str) -> EditResult:
    target = (WORKSPACE / path).resolve()
    if not target.is_relative_to(WORKSPACE):
        raise ToolError(f"refused: {path} is outside the workspace")
    text = target.read_text()
    if text.count(old) != 1:
        raise ToolError(f"refused: old appears {text.count(old)} times in {path}")
    target.write_text(text.replace(old, new, 1))
    return {"path": path, "replaced": 1}


@mcp.tool(name="progress_bar", description="Report progress in steps, pausing a random time between steps.")
async def progress_bar(ctx: Context, steps: int = 10, max_gap: float = 1.5) -> str:
    for i in range(1, steps + 1):
        await asyncio.sleep(random.uniform(0.1, max_gap))             # random time gap
        bar = "#" * i + "." * (steps - i)
        await ctx.report_progress(i, steps, f"[{bar}] {i}/{steps}")   # progress notification
    return f"done in {steps} steps"


if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)   # http://127.0.0.1:8000/mcp

## 3. Running them with the Inspector

- **Start the HTTP server first,** in a terminal, from the folder that holds `mcp-demo`:

In [22]:
%%bash
nohup python mcp-demo/server_http.py > mcp-demo/server_http.log 2>&1 < /dev/null &
echo $! > mcp-demo/server_http.pid
echo "Server started with PID $(cat mcp-demo/server_http.pid)"

Server started with PID 51554


### Write a catalog file for the Inspector

- **`catalog.json` lists the servers the Inspector can connect to,** under one key, `mcpServers`, in the same shape as the host configuration on slide 33.
- **Each entry has a name and a `type`:** `stdio`, `sse` or `streamable-http`. An entry with no `type`, or an unknown one, is treated as `stdio`.
- **A `stdio` entry is a command line:** `command` and `args`, and optionally `env` and `cwd`. The Inspector runs that command when you connect.
- **A `streamable-http` entry is a URL:** `url`, and optionally `headers`. The server must already be running.
- **An entry may set `protocolEra`** (`legacy`, `auto` or `modern`) to choose which protocol revision the Inspector speaks to that server.
- **`--catalog` opens the file for writing:** servers you add or edit in the Inspector are saved back to it, and a missing file is created empty. Running the cell below again overwrites those edits.
- **`--config` opens the same format read-only.** The two flags cannot be combined, and `--catalog` cannot be combined with a server named on the command line.
- **The cell writes absolute paths:** `sys.executable` is the notebook's `.venv` Python, which has `mcp` installed, and the server's full path works from any folder.
- **The catalog decides which programs the Inspector runs.** Whoever can write this file chooses the command, and slide 101 shows what happened when the Inspector's backend accepted such commands without authentication.

In [25]:
%%writefile mcp-demo/catalog.json

{
  "mcpServers": {
    "workspace-stdio": {
      "type": "stdio",
      "command": "/Users/rkrsn/COMSE6998-019/lectures/lecture-5--2026-10-07/mcp-demo/../.venv/bin/python",
      "args": ["/Users/rkrsn/COMSE6998-019/lectures/lecture-5--2026-10-07/mcp-demo/server_stdio.py"]
    },
    "workspace-http": {
      "type": "streamable-http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}


Overwriting mcp-demo/catalog.json


- **Open the Inspector with the catalog** in a second terminal (`catalog.json` is created by the next cell):

In [27]:
%%bash
npx @modelcontextprotocol/inspector --web --catalog mcp-demo/catalog.json

Starting MCP inspector...

MCP Inspector Web is up and running at:
   http://127.0.0.1:6274?MCP_INSPECTOR_API_TOKEN=ef5d94a84e036da2b863b7c4a3272fe2300536faedd53738b8a07d5b944f6378

   Sandbox (MCP Apps): http://127.0.0.1:6275/sandbox

   Auth token: ef5d94a84e036da2b863b7c4a3272fe2300536faedd53738b8a07d5b944f6378

   Secrets: OS keychain

Opening browser...
Process was interrupted.


CalledProcessError: Command 'b'npx @modelcontextprotocol/inspector --web --catalog mcp-demo/catalog.json\n'' returned non-zero exit status 0.